# Ingest AEMO P5MIN forecasts

This notebook downloads recent historical and current P5MIN files and extracts their CSVs into the forecast Bronze folders. It does not parse the forecast rows; the shared Silver notebook handles that step.

### Storage prerequisite

Before running the pipeline, I created the Databricks Volume `/Volumes/workspace/default/aemo_mlops_volume`. Forecast files remain separate from actual demand under its `bronze/forecast` folder.

## 1. Discover recent archive and current files

The archive listing contains one ZIP per day. The current listing contains one ZIP per five-minute forecast run. Reading both HTML listings avoids predicting filenames that change over time.

### Filename patterns

```python
archive: r"PUBLIC_P5MIN_\d{8}\.zip"
current: r"PUBLIC_P5MIN_\d{12}_\d{14}\.zip"
```

- `\d{8}` is the archive date in `YYYYMMDD` form.
- The current filename contains a twelve-digit interval time followed by a fourteen-digit publication timestamp.
- `\.zip` requires the literal ZIP extension.

Only the latest three months of daily archives are selected because each daily archive expands into 288 five-minute files.

In [ ]:
import os
import re
import time
import zipfile
from datetime import datetime
from urllib.parse import urljoin

import requests
from dateutil.relativedelta import relativedelta


archive_url = (
    "https://www.nemweb.com.au/"
    "REPORTS/ARCHIVE/P5_Reports/"
)
current_url = (
    "https://www.nemweb.com.au/"
    "REPORTS/CURRENT/P5_Reports/"
)


def discover_zip_urls(folder, filename_pattern):
    # Read the live directory listing rather than constructing filenames.
    response = requests.get(folder)
    response.raise_for_status()

    files = re.findall(filename_pattern, response.text)

    return [
        urljoin(folder, filename)
        for filename in sorted(set(files))
    ]


archive_urls = discover_zip_urls(
    archive_url,
    r"PUBLIC_P5MIN_\d{8}\.zip"
)

# The daily archive is large, so this implementation keeps only recent history.
start_date = datetime.today() - relativedelta(months=3)

archive_urls = [
    url
    for url in archive_urls
    if datetime.strptime(
        re.search(r"\d{8}", os.path.basename(url)).group(),
        "%Y%m%d"
    ) >= start_date
]

current_urls = discover_zip_urls(
    current_url,
    r"PUBLIC_P5MIN_\d{12}_\d{14}\.zip"
)

In [ ]:
print(f"Found {len(archive_urls)} recent daily archives")
print(f"Found {len(current_urls)} current five-minute files")

# Display the discovered URLs before downloading.
archive_urls, current_urls

## 2. Download only missing ZIPs

The source ZIPs are preserved unchanged in Bronze. Existing files are skipped so the notebook can be run again without downloading the same data twice.

In [ ]:
def download_if_not_exists(url, bronze_folder):
    filename = os.path.basename(url)
    path = os.path.join(bronze_folder, filename)

    if os.path.exists(path):
        print(f"Skipping: {filename}")
        return path

    print(f"Downloading: {filename}")
    start = time.time()

    response = requests.get(url)
    response.raise_for_status()

    with open(path, "wb") as file:
        file.write(response.content)

    print(f"Finished: {filename} - {time.time() - start:.1f} seconds")

    return path

In [ ]:
archive_folder = (
    "/Volumes/workspace/default/aemo_mlops_volume/"
    "bronze/forecast/archive"
)
current_folder = (
    "/Volumes/workspace/default/aemo_mlops_volume/"
    "bronze/forecast/current"
)

os.makedirs(archive_folder, exist_ok=True)
os.makedirs(current_folder, exist_ok=True)

for url in archive_urls:
    download_if_not_exists(url, archive_folder)

for url in current_urls:
    download_if_not_exists(url, current_folder)

## 3. Unpack the historical daily archives

Each historical daily ZIP contains 288 inner ZIPs—one for every five-minute forecast run. Each inner ZIP contains one P5MIN CSV. The two extraction levels leave only CSV files in `archive_uncompressed`.

In [ ]:
archive_csv_folder = archive_folder + "_uncompressed"

os.makedirs(archive_csv_folder, exist_ok=True)

archive_zip_files = [
    filename
    for filename in os.listdir(archive_folder)
    if filename.endswith(".zip")
]

start = time.time()
print(f"Found {len(archive_zip_files)} daily archive ZIPs")

# First level: one daily ZIP expands into five-minute ZIPs.
for i, filename in enumerate(sorted(archive_zip_files), 1):
    path = os.path.join(archive_folder, filename)

    with zipfile.ZipFile(path, "r") as zip_file:
        inner_zip_names = [
            member
            for member in zip_file.namelist()
            if member.lower().endswith(".zip")
        ]

        expected_csvs = [
            os.path.basename(os.path.splitext(member)[0] + ".CSV")
            for member in inner_zip_names
        ]

        already_extracted = expected_csvs and all(
            os.path.exists(os.path.join(archive_csv_folder, csv_name))
            for csv_name in expected_csvs
        )

        if already_extracted:
            print(f"[{i}/{len(archive_zip_files)}] Skipping: {filename}")
            continue

        print(f"[{i}/{len(archive_zip_files)}] Extracting: {filename}")
        zip_file.extractall(archive_csv_folder)


# Second level: each five-minute ZIP expands into one P5MIN CSV.
inner_zip_files = [
    filename
    for filename in os.listdir(archive_csv_folder)
    if filename.endswith(".zip")
]

print(f"Found {len(inner_zip_files)} inner ZIPs")

for i, filename in enumerate(sorted(inner_zip_files), 1):
    path = os.path.join(archive_csv_folder, filename)

    with zipfile.ZipFile(path, "r") as zip_file:
        members = zip_file.namelist()

        already_extracted = all(
            os.path.exists(
                os.path.join(archive_csv_folder, os.path.basename(member))
            )
            for member in members
        )

        if not already_extracted:
            zip_file.extractall(archive_csv_folder)

    # Keep the final folder directly readable by Spark by removing inner ZIPs.
    os.remove(path)
    print(f"[{i}/{len(inner_zip_files)}] Extracted: {filename}")

print(f"Finished archive extraction in {time.time() - start:.1f} seconds")

## 4. Unpack the current files

Current P5MIN ZIPs contain one CSV directly, so only one extraction level is required. Existing CSVs are skipped.

In [ ]:
current_csv_folder = current_folder + "_uncompressed"

os.makedirs(current_csv_folder, exist_ok=True)

current_zip_files = [
    filename
    for filename in os.listdir(current_folder)
    if filename.endswith(".zip")
]

start = time.time()
print(f"Found {len(current_zip_files)} current ZIPs")

for i, filename in enumerate(sorted(current_zip_files), 1):
    path = os.path.join(current_folder, filename)

    with zipfile.ZipFile(path, "r") as zip_file:
        members = zip_file.namelist()

        already_extracted = all(
            os.path.exists(
                os.path.join(current_csv_folder, os.path.basename(member))
            )
            for member in members
        )

        if already_extracted:
            print(f"[{i}/{len(current_zip_files)}] Skipping: {filename}")
            continue

        print(f"[{i}/{len(current_zip_files)}] Extracting: {filename}")
        zip_file.extractall(current_csv_folder)

print(f"Finished current extraction in {time.time() - start:.1f} seconds")